# Chapter 6: CUSUM, EWMA and capability on lager pH
150 synthetic pH results, seed 9, with a step of 0.015 pH (0.5 sigma) from result 61. Then the same step in 3,000 simulated series (seed 44) and the run length table from a Markov chain.

**No code tier.** `sheets/ch06-cusum.xlsx` holds the recipe with live formulas: baseline from results 1 to 60, a C column and an alarm flag. Change a pH value and watch the first alarm move.

Synthetic data from Kestrel Brewing, an invented plant. Copyright 2026 Ankur Napa. All rights reserved. You may use these files for your own learning; do not republish or redistribute them.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = "../data/"
from scipy.stats import norm

In [ ]:
x = pd.read_csv(DATA + "ch06_ph.csv").ph.to_numpy()
base = x[:60]; t = base.mean(); s = np.abs(np.diff(base)).mean() / 1.128
k, h = 0.5 * s, 5 * s
C, c = [], 0.0
for v in x: c = max(0.0, c + v - (t + k)); C.append(c)
alarm = next(i for i in range(60, 150) if C[i] > h) + 1
w = 3 * s * math.sqrt(0.2 / 1.8); Z, z = [], t
for v in x: z = 0.2 * v + 0.8 * z; Z.append(z)
ewma_out = next(i for i in range(60, 150) if abs(Z[i] - t) > w) + 1
print(f"centre {t:.3f} sigma {s:.4f} limits {t - 3 * s:.3f} to {t + 3 * s:.3f} | after the step mean {x[60:].mean():.3f}")
print(f"CUSUM alarm at result {alarm} (C = {C[alarm - 1]:.3f}) | EWMA leaves its limits at {ewma_out}")

In [ ]:
n = np.arange(1, 151)
fig, ax = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
ax[0].plot(n, x, lw=0.8, color="#2f7f72"); [ax[0].axhline(y, ls=st, color="#0b4f45") for y, st in ((t, "-"), (t + 3 * s, "--"), (t - 3 * s, "--"))]
ax[1].plot(n, C, color="#0b4f45"); ax[1].axhline(h, ls="--", color="#b23a2e"); ax[1].set_ylabel("CUSUM C")
ax[2].plot(n, x, ".", ms=2, color="#9bb8b2"); ax[2].plot(n, Z, color="#0b4f45"); [ax[2].axhline(t + d, ls="--", color="#6b6b6b") for d in (w, -w)]
for a in ax: a.axvline(61, color="#c27c0e", ls=":")
ax[2].set_xlabel("result (one per bright tank fill)"); plt.show()

In [ ]:
print(f"Cp {0.3 / (6 * s):.2f}  Cpk {(t - 4.2) / (3 * s):.2f}  Ppk {(t - 4.2) / (3 * base.std(ddof=1)):.2f}")
print(f"Cpk if the new level holds: {(4.5 - x[60:].mean()) / (3 * s):.2f}  | share of variance from the test {0.02 ** 2 / s ** 2:.0%}")

## The 3,000 run test
Each series has its own baseline from its first 60 results, as a lab would.

In [ ]:
rng = np.random.default_rng(44)
sims = rng.normal(4.35, 0.03, (3000, 150)); sims[:, 60:] += 0.015
cl = sims[:, :60].mean(1, keepdims=True); sg = np.abs(np.diff(sims[:, :60], axis=1)).mean(1, keepdims=True) / 1.128
def first(m): return np.where(m.any(1), m.argmax(1) + 1, 0)
side = np.sign(sims - cl); run = np.ones(3000); r8 = np.zeros((3000, 90), bool); cc = np.zeros(3000); cs = np.zeros((3000, 90), bool)
for i in range(150):
    if i: run = np.where(side[:, i] == side[:, i - 1], run + 1, 1)
    cc = np.maximum(0, cc + sims[:, i] - (cl[:, 0] + 0.5 * sg[:, 0]))
    if i >= 60: r8[:, i - 60] = run >= 8; cs[:, i - 60] = cc > 5 * sg[:, 0]
for name, f in (("one point", first(np.abs(sims[:, 60:] - cl) > 3 * sg)), ("eight in a row", first(r8)), ("CUSUM", first(cs))):
    print(f"{name:15s} caught in {(f > 0).mean():.0%} of runs, median {int(np.median(f[f > 0]))} results after the step")

## Average run length by Markov chain
The table in 6.1. The CUSUM and EWMA chains split the chart statistic into small cells.

In [ ]:
def arl_point(d): return 1 / (1 - norm.cdf(3 - d) + norm.cdf(-3 - d))

def arl_eight(d, n=8):
    up, dn = norm.cdf(d), norm.cdf(-d); m = 2 * (n - 1); Q = np.zeros((m, m))
    for st in range(m):
        side, r = divmod(st, n - 1)
        if side == 0:
            if r + 1 < n - 1: Q[st, r + 1] = up
            Q[st, n - 1] = dn
        else:
            if r + 1 < n - 1: Q[st, n + r] = dn
            Q[st, 0] = up
    T = np.linalg.solve(np.eye(m) - Q, np.ones(m)); return 1 + up * T[0] + dn * T[n - 1]

def arl_cusum(d, k=0.5, h=5.0, m=300):
    w = 2 * h / (2 * m - 1)
    def one(dd):
        Q = np.empty((m, m))
        for i in range(m):
            up = norm.cdf((np.arange(m) + 0.5) * w - i * w + k - dd); Q[i] = np.diff(np.r_[0.0, up]); Q[i, 0] = up[0]
        return np.linalg.solve(np.eye(m) - Q, np.ones(m))[0]
    return 1 / (1 / one(d) + 1 / one(-d))

def arl_ewma(d, lam=0.2, L=3.0, m=401):
    lim = L * math.sqrt(lam / (2 - lam)); w = 2 * lim / m; mid = -lim + (np.arange(m) + 0.5) * w
    Q = np.array([norm.cdf((mid + w / 2 - (1 - lam) * z) / lam - d) - norm.cdf((mid - w / 2 - (1 - lam) * z) / lam - d) for z in mid])
    return np.linalg.solve(np.eye(m) - Q, np.ones(m))[m // 2]

shifts = (0, 0.3, 0.5, 1.0)
pd.DataFrame({f: [round(fn(d), 1) for d in shifts] for f, fn in (("one point", arl_point), ("eight in a row", arl_eight), ("CUSUM", arl_cusum), ("EWMA", arl_ewma))},
             index=[f"{d} sigma" for d in shifts]).T

**What to look at.** No single pH result crosses a limit, yet the CUSUM alarms at 94 and the EWMA at 93. Cpk drops from 1.47 to 1.31 with nothing failed. Check the buffer chart before blaming the beer.